#### Load Processed Dataset
Load the enriched transaction dataset prepared in the previous notebooks. It contains transaction-level discount measures and product attributes required for statistical analysis.

In [1]:
import pandas as pd
import numpy as np

transactions_enriched = pd.read_csv(
    "../data/processed/transactions_enriched.csv",
    low_memory=False
)

print("Dataset shape:", transactions_enriched.shape)
transactions_enriched.head()

Dataset shape: (2595732, 24)


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,...,promotion_records,retail_discount_amount,coupon_discount_amount,coupon_match_discount_amount,total_discount,estimated_original_value,discount_percentage,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,...,NaN,0.60,-0.0,-0.0,0.60,1.99,30.150754,POTATOES,POTATOES RUSSET (BULK&BAG),Private
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,...,NaN,-0.00,-0.0,-0.0,0.00,0.82,0.000000,ONIONS,ONIONS SWEET (BULK&BAG),National
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,...,NaN,0.30,-0.0,-0.0,0.30,1.29,23.255814,VEGETABLES - ALL OTHERS,CELERY,Private
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,...,NaN,-0.00,-0.0,-0.0,0.00,1.21,0.000000,TROPICAL FRUIT,BANANAS,National
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,...,NaN,0.39,-0.0,-0.0,0.39,1.89,20.634921,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,Private


#### Prepare Statistical Analysis Variables
Create the categorical variables required to compare purchasing behaviour across discount groups.

In [2]:
# Identify whether a transaction record received a discount
transactions_enriched["discount_status"] = (
    transactions_enriched["total_discount"]
    .gt(0)
    .map({True: "Discounted", False: "Non-Discounted"})
)

# Identify multi-unit purchases
transactions_enriched["purchase_type"] = np.where(
    transactions_enriched["QUANTITY"] > 1,
    "Multi-Unit",
    "Single-Unit"
)

# Check the resulting categories
print("Discount status:")
print(transactions_enriched["discount_status"].value_counts())

print("\nPurchase type:")
print(transactions_enriched["purchase_type"].value_counts())

print("\nHouseholds:", transactions_enriched["household_key"].nunique())

Discount status:
discount_status
Discounted        1319391
Non-Discounted    1276341
Name: count, dtype: int64

Purchase type:
purchase_type
Single-Unit    2065433
Multi-Unit      530299
Name: count, dtype: int64

Households: 2500


#### Examine the Relationship Between Discount Status and Purchase Type
Create a contingency table to compare single-unit and multi-unit purchases across discounted and non-discounted transaction records.

In [3]:
# Create a contingency table
purchase_contingency = pd.crosstab(
    transactions_enriched["discount_status"],
    transactions_enriched["purchase_type"]
)

purchase_contingency

purchase_type,Multi-Unit,Single-Unit
discount_status,,
Discounted,320325,999066
Non-Discounted,209974,1066367


#### Aggregate Purchase Counts by Household
Summarise discounted and non-discounted single-unit and multi-unit purchases for each household so that repeated transactions can be handled appropriately during statistical inference.

In [4]:
# Count each purchase category separately for every household
household_purchase_counts = (
    transactions_enriched
    .groupby(["household_key", "discount_status", "purchase_type"])
    .size()
    .unstack(["discount_status", "purchase_type"], fill_value=0)
)

household_purchase_counts.head()

discount_status Discounted             Non-Discounted            
purchase_type   Multi-Unit Single-Unit     Multi-Unit Single-Unit
household_key                                                    
1                      120         693             93         821
2                       57         272             25         360
3                      232         306            121         263
4                       26         104             30         141
5                       14         104              7          97

#### Estimate the Confidence Interval Using Household Bootstrap
Resample households with replacement and calculate the difference in multi-unit purchase rates for each resample. Use the resulting distribution to estimate a 95% confidence interval.

In [5]:
# Extract household-level purchase counts in a fixed order
household_counts = household_purchase_counts[
    [
        ("Discounted", "Multi-Unit"),
        ("Discounted", "Single-Unit"),
        ("Non-Discounted", "Multi-Unit"),
        ("Non-Discounted", "Single-Unit")
    ]
].to_numpy(dtype=float)

# Calculate the observed difference in multi-unit purchase rates
total_counts = household_counts.sum(axis=0)

discounted_rate = (
    total_counts[0] / (total_counts[0] + total_counts[1])
)

non_discounted_rate = (
    total_counts[2] / (total_counts[2] + total_counts[3])
)

observed_difference = (discounted_rate - non_discounted_rate) * 100

# Bootstrap by resampling households
rng = np.random.default_rng(42)
n_households = len(household_counts)
n_bootstrap = 2000

bootstrap_differences = []

for _ in range(n_bootstrap):
    sampled_indices = rng.integers(0, n_households, n_households)
    sampled_counts = household_counts[sampled_indices].sum(axis=0)

    discounted_rate_sample = (
        sampled_counts[0] /
        (sampled_counts[0] + sampled_counts[1])
    )

    non_discounted_rate_sample = (
        sampled_counts[2] /
        (sampled_counts[2] + sampled_counts[3])
    )

    difference = (
        discounted_rate_sample - non_discounted_rate_sample
    ) * 100

    bootstrap_differences.append(difference)

# Calculate the 95% confidence interval
confidence_interval = np.percentile(
    bootstrap_differences, [2.5, 97.5]
)

print(f"Discounted multi-unit rate: {discounted_rate:.2%}")
print(f"Non-discounted multi-unit rate: {non_discounted_rate:.2%}")
print(f"Observed difference: {observed_difference:.2f} percentage points")
print(
    f"95% Bootstrap CI: "
    f"({confidence_interval[0]:.2f}, {confidence_interval[1]:.2f})"
)

Discounted multi-unit rate: 24.28%
Non-discounted multi-unit rate: 16.45%
Observed difference: 7.83 percentage points
95% Bootstrap CI: (7.48, 8.20)


#### Check Household Eligibility for Within-Household Analysis
Identify households with both discounted and non-discounted transaction records. These households can contribute to a stratified comparison of purchase behaviour.

In [6]:
# Check whether each household has records in both discount groups
household_discount_presence = (
    transactions_enriched
    .groupby("household_key")["discount_status"]
    .nunique()
)

# Count households with both groups
eligible_households = household_discount_presence[
    household_discount_presence == 2
]

print("Total households:", len(household_discount_presence))
print("Households with both discount groups:", len(eligible_households))
print(
    "Households with only one discount group:",
    (household_discount_presence == 1).sum()
)

Total households: 2500
Households with both discount groups: 2500
Households with only one discount group: 0


#### Apply the Cochran–Mantel–Haenszel Test
Construct a 2×2 contingency table for each household and apply the CMH test to evaluate the common association between discount status and multi-unit purchasing.

In [7]:
from statsmodels.stats.contingency_tables import StratifiedTable

# Create one 2x2 table per household
household_tables = []

for household in household_purchase_counts.index:
    counts = household_purchase_counts.loc[household]

    table = [
        [
            counts[("Discounted", "Multi-Unit")],
            counts[("Discounted", "Single-Unit")]
        ],
        [
            counts[("Non-Discounted", "Multi-Unit")],
            counts[("Non-Discounted", "Single-Unit")]
        ]
    ]

    household_tables.append(table)

# Apply the CMH test
cmh_result = StratifiedTable(household_tables)

test_result = cmh_result.test_null_odds()

print("CMH test statistic:", test_result.statistic)
print("P-value:", test_result.pvalue)
print("Common odds ratio:", cmh_result.oddsratio_pooled)
print("95% CI for common odds ratio:", cmh_result.oddsratio_pooled_confint())

CMH test statistic: 21964.00007901658
P-value: 0.0
Common odds ratio: 1.6108230143755466
95% CI for common odds ratio: (1.6006407800761155, 1.621070021418881)


#### RQ1: Statistical Findings and Interpretation

**Research Question:** Is discount status associated with multi-unit purchasing?

The initial comparison showed that 24.28% of discounted transaction records involved multi-unit purchases, compared with 16.45% of non-discounted records. This represents a difference of 7.83 percentage points.

To account for repeated transactions from the same households, a household-level bootstrap was performed using 2,000 resamples. The estimated 95% confidence interval for the difference was 7.48 to 8.20 percentage points.

A Cochran–Mantel–Haenszel (CMH) test was then performed across all 2,500 households. The test produced a statistic of 21,964.00, with a p-value below numerical reporting precision (p < 0.001). The common odds ratio was 1.611, with a 95% confidence interval of 1.601 to 1.621.

**Conclusion:** The results provide strong evidence of a positive association between discount status and multi-unit purchasing, even after accounting for household differences. Discounted transaction records had approximately 61.1% higher odds of multi-unit purchasing than non-discounted records.

However, this is an observational analysis and does not establish that discounts directly cause customers to purchase more units.

In [8]:
#########

#### Create Discount Percentage Bands
Group transaction records into discount ranges to examine how purchasing behaviour varies with discount size. Records with undefined discount percentages are excluded from this band-based analysis.

In [9]:
# Define discount ranges
discount_bins = [-0.01, 0, 10, 25, 50, 100]
discount_labels = [
    "No Discount",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

# Create discount bands
transactions_enriched["discount_band"] = pd.cut(
    transactions_enriched["discount_percentage"],
    bins=discount_bins,
    labels=discount_labels
)

# Check the number of records in each band
print(transactions_enriched["discount_band"].value_counts().sort_index())

print(
    "\nRecords excluded due to undefined discount percentage:",
    transactions_enriched["discount_band"].isna().sum()
)

discount_band
No Discount    1266169
Low             169606
Moderate        494277
High            574626
Very High        80882
Name: count, dtype: int64

Records excluded due to undefined discount percentage: 10172


#### Compare Multi-Unit Purchasing Across Discount Bands
Calculate the number and percentage of multi-unit purchases within each discount band to examine how purchasing behaviour varies with discount size.

In [10]:
# Calculate purchase counts by discount band and purchase type
discount_purchase_summary = pd.crosstab(
    transactions_enriched["discount_band"],
    transactions_enriched["purchase_type"]
)

# Calculate percentage of each purchase type within each band
discount_purchase_percentage = (
    discount_purchase_summary
    .div(discount_purchase_summary.sum(axis=1), axis=0)
    * 100
)

print("Purchase counts:")
display(discount_purchase_summary)

print("\nPurchase percentages:")
display(discount_purchase_percentage.round(2))

Purchase counts:


purchase_type,Multi-Unit,Single-Unit
discount_band,,
No Discount,209954,1056215
Low,50545,119061
Moderate,96167,398110
High,151286,423340
Very High,22327,58555



Purchase percentages:


purchase_type,Multi-Unit,Single-Unit
discount_band,,
No Discount,16.58,83.42
Low,29.80,70.20
Moderate,19.46,80.54
High,26.33,73.67
Very High,27.60,72.40


#### Aggregate Purchases by Household and Discount Band
Count multi-unit and single-unit purchases for each household within each discount band. This prepares the data for statistical comparisons that account for repeated household observations.

In [11]:
# Aggregate purchase counts by household and discount band
household_discount_counts = (
    transactions_enriched
    .groupby(["household_key", "discount_band", "purchase_type"], observed=True)
    .size()
    .unstack("purchase_type", fill_value=0)
    .reset_index()
)

# Calculate the total number of records in each household-band group
household_discount_counts["total_purchases"] = (
    household_discount_counts["Multi-Unit"]
    + household_discount_counts["Single-Unit"]
)

# Inspect the aggregated data
print("Aggregated shape:", household_discount_counts.shape)
display(household_discount_counts.head())

print("\nHouseholds represented in each discount band:")
print(
    household_discount_counts
    .groupby("discount_band", observed=True)["household_key"]
    .nunique()
)

Aggregated shape: (12368, 5)


purchase_type,household_key,discount_band,Multi-Unit,Single-Unit,total_purchases
0,1,No Discount,93,811,904
1,1,Low,5,73,78
2,1,Moderate,41,300,341
3,1,High,68,287,355
4,1,Very High,6,33,39



Households represented in each discount band:
discount_band
No Discount    2500
Low            2472
Moderate       2497
High           2493
Very High      2406
Name: household_key, dtype: int64


#### Fit a Household-Clustered Logistic Model
Fit a binomial GEE model using household as the clustering unit. Use the multi-unit purchase proportion and total purchases as weights to account for the aggregated counts.

In [12]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Calculate the proportion of multi-unit purchases
household_discount_counts["multi_unit_rate"] = (
    household_discount_counts["Multi-Unit"]
    / household_discount_counts["total_purchases"]
)

# Fit a binomial GEE model with household-level clustering
gee_model = smf.gee(
    "multi_unit_rate ~ C(discount_band)",
    groups="household_key",
    data=household_discount_counts,
    weights=household_discount_counts["total_purchases"],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

gee_result = gee_model.fit()

print(gee_result.summary())

                               GEE Regression Results                              
Dep. Variable:             multi_unit_rate   No. Observations:                12368
Model:                                 GEE   No. clusters:                     2500
Method:                        Generalized   Min. cluster size:                   2
                      Estimating Equations   Max. cluster size:                   5
Family:                           Binomial   Mean cluster size:                 4.9
Dependence structure:         Exchangeable   Num. iterations:                     4
Date:                     Thu, 12 Nov 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         14:33:25
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                        -1.6156        

/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_estimating_equations.py:1842: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(np.diag(self.cov_robust))


#### Validate GEE Estimates and Test the Overall Discount-Band Effect
Inspect the model's covariance estimates to investigate the NaN standard error, then perform an overall Wald test of the four discount-band coefficients.

In [13]:
# Inspect the estimated covariance matrix
print("Covariance matrix:")
display(gee_result.cov_params())

print("\nStandard errors:")
print(gee_result.bse)

# Test whether all discount-band coefficients are jointly zero
overall_test = gee_result.wald_test(
    "C(discount_band)[T.Low] = 0, "
    "C(discount_band)[T.Moderate] = 0, "
    "C(discount_band)[T.High] = 0, "
    "C(discount_band)[T.Very High] = 0"
)

print("\nOverall Wald test:")
print("Test statistic:", overall_test.statistic)
print("P-value:", overall_test.pvalue)

Covariance matrix:


,Intercept,C(discount_band)[T.Low],C(discount_band)[T.Moderate],C(discount_band)[T.High],C(discount_band)[T.Very High]
Intercept,-0.000104,-0.000243,-0.000081,-0.000048,-0.000515
C(discount_band)[T.Low],0.000110,0.000408,0.000058,0.000037,0.000055
C(discount_band)[T.Moderate],-0.000021,0.000036,0.000119,0.000076,-0.000002
C(discount_band)[T.High],-0.000002,0.000034,0.000075,0.000130,0.000074
C(discount_band)[T.Very High],0.000089,0.000003,0.000045,0.000103,0.000279



Standard errors:
Intercept                             NaN
C(discount_band)[T.Low]          0.020199
C(discount_band)[T.Moderate]     0.010917
C(discount_band)[T.High]         0.011396
C(discount_band)[T.Very High]    0.016717
dtype: float64

Overall Wald test:
Test statistic: [[4139.38144193]]
P-value: 0.0


/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


#### Refit the GEE Model Using an Independence Structure
Refit the binomial GEE model with an independence working correlation structure while retaining household-level clustering and robust covariance estimation.

In [14]:
# Refit the model using an independence working correlation structure
gee_model_ind = smf.gee(
    "multi_unit_rate ~ C(discount_band)",
    groups="household_key",
    data=household_discount_counts,
    weights=household_discount_counts["total_purchases"],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence()
)

gee_result_ind = gee_model_ind.fit()

print(gee_result_ind.summary())

                               GEE Regression Results                              
Dep. Variable:             multi_unit_rate   No. Observations:                12368
Model:                                 GEE   No. clusters:                     2500
Method:                        Generalized   Min. cluster size:                   2
                      Estimating Equations   Max. cluster size:                   5
Family:                           Binomial   Mean cluster size:                 4.9
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Thu, 12 Nov 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         14:33:25
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                        -1.6156      0.

#### Validate the Independence GEE Model
Inspect the robust standard errors and covariance matrix, then perform an overall Wald test to determine whether discount band is associated with multi-unit purchasing.

In [15]:
# Inspect standard errors and covariance matrix
print("Standard errors:")
print(gee_result_ind.bse)

print("\nCovariance matrix:")
display(gee_result_ind.cov_params())

# Overall test: all discount-band coefficients equal zero
overall_test_ind = gee_result_ind.wald_test(
    "C(discount_band)[T.Low] = 0, "
    "C(discount_band)[T.Moderate] = 0, "
    "C(discount_band)[T.High] = 0, "
    "C(discount_band)[T.Very High] = 0"
)

print("\nOverall Wald test:")
print("Test statistic:", overall_test_ind.statistic)
print("P-value:", overall_test_ind.pvalue)

Standard errors:
Intercept                        0.012840
C(discount_band)[T.Low]          0.020579
C(discount_band)[T.Moderate]     0.011270
C(discount_band)[T.High]         0.011855
C(discount_band)[T.Very High]    0.018359
dtype: float64

Covariance matrix:


,Intercept,C(discount_band)[T.Low],C(discount_band)[T.Moderate],C(discount_band)[T.High],C(discount_band)[T.Very High]
Intercept,0.000165,-0.000072,-0.000035,-0.000047,-0.000046
C(discount_band)[T.Low],-0.000072,0.000424,0.000064,0.000061,0.000043
C(discount_band)[T.Moderate],-0.000035,0.000064,0.000127,0.000083,0.000058
C(discount_band)[T.High],-0.000047,0.000061,0.000083,0.000141,0.000127
C(discount_band)[T.Very High],-0.000046,0.000043,0.000058,0.000127,0.000337



Overall Wald test:
Test statistic: [[3581.90645679]]
P-value: 0.0


/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


#### Pairwise Comparisons Between Discount Bands
Compare multi-unit purchasing odds between every pair of discount bands using the fitted GEE model. Apply the Holm correction to account for multiple statistical comparisons.

In [16]:
from itertools import combinations
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# Get model coefficients and robust covariance matrix
params = gee_result_ind.params
cov_matrix = gee_result_ind.cov_params()
parameter_names = list(params.index)

# No Discount is the reference category
band_coefficients = {
    "No Discount": 0.0,
    "Low": params["C(discount_band)[T.Low]"],
    "Moderate": params["C(discount_band)[T.Moderate]"],
    "High": params["C(discount_band)[T.High]"],
    "Very High": params["C(discount_band)[T.Very High]"]
}

bands = list(band_coefficients.keys())
pairwise_results = []

# Compare all possible pairs
for band_a, band_b in combinations(bands, 2):

    contrast = np.zeros(len(params))

    if band_a != "No Discount":
        contrast[parameter_names.index(
            f"C(discount_band)[T.{band_a}]"
        )] = 1

    if band_b != "No Discount":
        contrast[parameter_names.index(
            f"C(discount_band)[T.{band_b}]"
        )] = -1

    log_odds_difference = (
        band_coefficients[band_a] - band_coefficients[band_b]
    )

    standard_error = np.sqrt(
        contrast @ cov_matrix.to_numpy() @ contrast
    )

    z_value = log_odds_difference / standard_error
    p_value = 2 * norm.sf(abs(z_value))

    odds_ratio = np.exp(log_odds_difference)
    lower_ci = np.exp(log_odds_difference - 1.96 * standard_error)
    upper_ci = np.exp(log_odds_difference + 1.96 * standard_error)

    pairwise_results.append({
        "Comparison": f"{band_a} vs {band_b}",
        "Odds Ratio": odds_ratio,
        "95% CI Lower": lower_ci,
        "95% CI Upper": upper_ci,
        "P-value": p_value
    })

# Apply Holm correction
pairwise_df = pd.DataFrame(pairwise_results)

pairwise_df["Holm Adjusted P-value"] = multipletests(
    pairwise_df["P-value"],
    method="holm"
)[1]

display(pairwise_df.round(5))

,Comparison,Odds Ratio,95% CI Lower,95% CI Upper,P-value,Holm Adjusted P-value
0,No Discount vs Low,0.46823,0.44972,0.48751,0.00000,0.00000
1,No Discount vs Moderate,0.82290,0.80492,0.84128,0.00000,0.00000
2,No Discount vs High,0.55624,0.54346,0.56932,0.00000,0.00000
3,No Discount vs Very High,0.52132,0.50290,0.54042,0.00000,0.00000
4,Low vs Moderate,1.75746,1.68805,1.82973,0.00000,0.00000
5,Low vs High,1.18795,1.14003,1.23789,0.00000,0.00000
6,Low vs Very High,1.11338,1.05814,1.17150,0.00004,0.00004
7,Moderate vs High,0.67595,0.66271,0.68945,0.00000,0.00000
8,Moderate vs Very High,0.63351,0.61078,0.65709,0.00000,0.00000
9,High vs Very High,0.93722,0.91019,0.96506,0.00001,0.00003


#### RQ2: Statistical Findings and Interpretation

**Research Question:** Does purchasing behaviour differ across discount levels?

The descriptive analysis showed that the multi-unit purchase rate varied across discount bands. The highest rate was observed in the Low discount band (29.80%), while the No Discount band had the lowest rate (16.58%).

A binomial Generalized Estimating Equations (GEE) model was fitted using household as the clustering unit. An independence working correlation structure was selected after the exchangeable model produced an invalid covariance estimate.

The overall Wald test showed a statistically significant association between discount band and multi-unit purchasing (Wald statistic = 3581.91, p < 0.001).

Pairwise comparisons were performed using odds ratios, with Holm correction for multiple testing. All ten pairwise comparisons remained statistically significant after correction (adjusted p < 0.05).

The Low discount band showed the highest estimated odds of multi-unit purchasing. Its odds were approximately 2.14 times those of the No Discount group. The Very High discount band had approximately 1.92 times the odds of the No Discount group.

**Conclusion:** Purchasing behaviour differs significantly across discount levels. However, the relationship is not linear, as the highest discount level does not produce the highest multi-unit purchasing rate. These findings indicate an association and do not establish a causal effect of discounts.

In [17]:
########

#### Prepare Data for Brand Comparison

Compare multi-unit purchasing behaviour between National and Private brands to examine whether the relationship between discounts and multi-unit purchases differs across brand types. Exclude FUEL and COUPON/MISC ITEMS, following the exclusions used in the earlier brand analysis.

In [18]:
rq3_data = transactions_enriched[
    ~transactions_enriched["COMMODITY_DESC"].isin(
        ["FUEL", "COUPON/MISC ITEMS"]
    )
].copy()

print("Dataset shape:", rq3_data.shape)
print(rq3_data["BRAND"].value_counts(dropna=False))

Dataset shape: (2566095, 27)
BRAND
National    1849277
Private      716818
Name: count, dtype: int64


#### Compare Multi-Unit Purchasing Across Brand Types

Calculate the number and percentage of multi-unit purchases for National and Private brands, separately for discounted and non-discounted transactions. This comparison will help us understand the differences in purchasing behaviour before conducting the statistical test.

In [19]:
rq3_summary = (
    rq3_data
    .groupby(["BRAND", "discount_status"], observed=True)
    .agg(
        total_purchases=("purchase_type", "size"),
        multi_unit_purchases=(
            "purchase_type",
            lambda x: (x == "Multi-Unit").sum()
        )
    )
    .reset_index()
)

rq3_summary["multi_unit_rate"] = (
    rq3_summary["multi_unit_purchases"]
    / rq3_summary["total_purchases"]
    * 100
)

rq3_summary

,BRAND,discount_status,total_purchases,multi_unit_purchases,multi_unit_rate
0,National,Discounted,885348,198178,22.384192
1,National,Non-Discounted,963929,157263,16.314791
2,Private,Discounted,410893,99003,24.094594
3,Private,Non-Discounted,305925,52055,17.015608


#### Aggregate Purchasing Data at Household Level

Since each household contributes multiple transaction records, we aggregate the data by household, brand type and discount status. We calculate the number of multi-unit and total purchases for each group. This will allow us to account for repeated observations from the same household in the statistical model.

In [20]:
rq3_household_counts = (
    rq3_data
    .groupby(
        ["household_key", "BRAND", "discount_status", "purchase_type"]
    )
    .size()
    .unstack("purchase_type", fill_value=0)
    .reset_index()
)

rq3_household_counts["total_purchases"] = (
    rq3_household_counts["Multi-Unit"]
    + rq3_household_counts["Single-Unit"]
)

rq3_household_counts["multi_unit_rate"] = (
    rq3_household_counts["Multi-Unit"]
    / rq3_household_counts["total_purchases"]
)

print("Shape:", rq3_household_counts.shape)
rq3_household_counts.head()

Shape: (9978, 7)


purchase_type,household_key,BRAND,discount_status,Multi-Unit,Single-Unit,total_purchases,multi_unit_rate
0,1,National,Discounted,113,594,707,0.159830
1,1,National,Non-Discounted,93,724,817,0.113831
2,1,Private,Discounted,7,99,106,0.066038
3,1,Private,Non-Discounted,0,95,95,0.000000
4,2,National,Discounted,46,209,255,0.180392


#### Check Household Coverage Across Brand and Discount Groups

Check whether households have records in all four combinations of brand type and discount status. This helps us understand the completeness of the household-level data before fitting the statistical model.

In [21]:
rq3_household_coverage = (
    rq3_household_counts
    .groupby("household_key")
    .size()
)

print("Total households:", rq3_household_coverage.size)
print(
    "Households with all 4 combinations:",
    (rq3_household_coverage == 4).sum()
)
print(
    "Households with fewer than 4 combinations:",
    (rq3_household_coverage < 4).sum()
)

print("\nNumber of household groups:")
print(rq3_household_coverage.value_counts().sort_index())

Total households: 2500
Households with all 4 combinations: 2483
Households with fewer than 4 combinations: 17

Number of household groups:
2       5
3      12
4    2483
Name: count, dtype: int64


#### Fit a Household-Clustered GEE Model

Fit a Generalized Estimating Equations (GEE) model to examine the relationship between discount status, brand type and multi-unit purchasing. Include an interaction term between discount status and brand type to test whether the discount association differs between National and Private brands. Use household as the clustering variable to account for repeated observations and robust standard errors to account for within-household dependence.

In [22]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set reference categories
rq3_household_counts["discount_status"] = pd.Categorical(
    rq3_household_counts["discount_status"],
    categories=["Non-Discounted", "Discounted"]
)

rq3_household_counts["BRAND"] = pd.Categorical(
    rq3_household_counts["BRAND"],
    categories=["National", "Private"]
)

# Fit GEE model with household-level clustering
gee_rq3 = smf.gee(
    "multi_unit_rate ~ C(discount_status) * C(BRAND)",
    groups="household_key",
    data=rq3_household_counts,
    weights=rq3_household_counts["total_purchases"],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence()
)

gee_rq3_result = gee_rq3.fit()

print(gee_rq3_result.summary())

                               GEE Regression Results                              
Dep. Variable:             multi_unit_rate   No. Observations:                 9978
Model:                                 GEE   No. clusters:                     2500
Method:                        Generalized   Min. cluster size:                   2
                      Estimating Equations   Max. cluster size:                   4
Family:                           Binomial   Mean cluster size:                 4.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Thu, 12 Nov 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         14:33:27
                                                           coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------
In

#### Calculate Odds Ratios for Each Brand Type

Convert the GEE model coefficients into odds ratios to measure the association between discounts and multi-unit purchasing for National and Private brands separately. Calculate 95% confidence intervals to understand the precision of these estimates.

In [23]:
from scipy.stats import norm

params = gee_rq3_result.params
cov = gee_rq3_result.cov_params()

# National brand discount effect
national_coef = params["C(discount_status)[T.Discounted]"]
national_se = np.sqrt(
    cov.loc[
        "C(discount_status)[T.Discounted]",
        "C(discount_status)[T.Discounted]"
    ]
)

# Private brand discount effect
private_coef = (
    params["C(discount_status)[T.Discounted]"]
    + params["C(discount_status)[T.Discounted]:C(BRAND)[T.Private]"]
)

private_vector = np.array([0, 1, 0, 1])
private_se = np.sqrt(
    private_vector @ cov.values @ private_vector
)

# Interaction effect
interaction_coef = params[
    "C(discount_status)[T.Discounted]:C(BRAND)[T.Private]"
]
interaction_se = np.sqrt(
    cov.loc[
        "C(discount_status)[T.Discounted]:C(BRAND)[T.Private]",
        "C(discount_status)[T.Discounted]:C(BRAND)[T.Private]"
    ]
)

# Calculate odds ratios and confidence intervals
rq3_effects = pd.DataFrame({
    "Brand": ["National", "Private", "Private vs National interaction"],
    "Odds_Ratio": np.exp([
        national_coef,
        private_coef,
        interaction_coef
    ]),
    "CI_Lower": np.exp([
        national_coef - 1.96 * national_se,
        private_coef - 1.96 * private_se,
        interaction_coef - 1.96 * interaction_se
    ]),
    "CI_Upper": np.exp([
        national_coef + 1.96 * national_se,
        private_coef + 1.96 * private_se,
        interaction_coef + 1.96 * interaction_se
    ])
})

rq3_effects

,Brand,Odds_Ratio,CI_Lower,CI_Upper
0,National,1.479307,1.444491,1.514963
1,Private,1.548089,1.494685,1.603400
2,Private vs National interaction,1.046496,1.004003,1.090787


#### Test the Brand Interaction Effect

Perform a Wald test of the interaction coefficient to determine whether the discount association differs significantly between National and Private brands. This is the main statistical test for RQ3.

In [24]:
interaction_test = gee_rq3_result.wald_test(
    "C(discount_status)[T.Discounted]:C(BRAND)[T.Private] = 0"
)

print("Wald statistic:", float(interaction_test.statistic))
print("P-value:", float(interaction_test.pvalue))

Wald statistic: 4.617708094361833
P-value: 0.031643492228568715


/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/tmp/ipykernel_2621980/1250995000.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("Wald statistic:", float(interaction_test.statistic))


#### Interpretation of RQ3 Results

The GEE model shows that discounted transactions are associated with higher odds of multi-unit purchasing for both National and Private brands. The odds ratio is 1.48 for National brands and 1.55 for Private brands.

The interaction effect is statistically significant (Wald statistic = 4.62, p = 0.032). Its odds ratio is 1.046, indicating that the discount-associated odds are approximately 4.65% higher for Private brands compared with National brands.

Therefore, the relationship between discounts and multi-unit purchasing differs significantly between the two brand types, with a slightly stronger association for Private brands. However, these findings indicate association and do not establish causation.

In [25]:
########

#### Prepare Data for Commodity Category Analysis

Examine the commodity categories available in the transaction dataset and their transaction counts. This will help us understand the distribution of purchases across categories and identify categories suitable for the statistical analysis.

In [26]:
rq4_data = transactions_enriched[
    ~transactions_enriched["COMMODITY_DESC"].isin(
        ["FUEL", "COUPON/MISC ITEMS"]
    )
].copy()

print("Dataset shape:", rq4_data.shape)

print("\nNumber of commodity categories:")
print(rq4_data["COMMODITY_DESC"].nunique())

print("\nTop 20 commodity categories:")
print(rq4_data["COMMODITY_DESC"].value_counts().head(20))

Dataset shape: (2566095, 27)

Number of commodity categories:
306

Top 20 commodity categories:
COMMODITY_DESC
SOFT DRINKS                  117532
FLUID MILK PRODUCTS           85630
BAKED BREAD/BUNS/ROLLS        83232
CHEESE                        74885
BAG SNACKS                    67190
FRZN MEAT/MEAT DINNERS        56064
BEEF                          48726
SOUP                          46135
YOGURT                        44697
FROZEN PIZZA                  43362
VEGETABLES - SHELF STABLE     41612
COLD CEREAL                   37870
CANDY - CHECKLANE             35556
LUNCHMEAT                     35162
TROPICAL FRUIT                34442
CANDY - PACKAGED              34177
REFRGRATD JUICES/DRNKS        31469
CRACKERS/MISC BKD FD          29723
CANNED JUICES                 29149
EGGS                          28420
Name: count, dtype: int64


#### Examine Commodity Category Distribution

Calculate the number of transactions in each commodity category and examine the distribution of category sizes. This will help identify categories with very few observations that may not provide reliable estimates in the statistical analysis.

In [27]:
commodity_counts = (
    rq4_data["COMMODITY_DESC"]
    .value_counts()
)

print("Category count summary:")
print(commodity_counts.describe())

print("\nCategories with fewer than 100 transactions:")
print((commodity_counts < 100).sum())

print("\nCategories with fewer than 500 transactions:")
print((commodity_counts < 500).sum())

print("\nCategories with fewer than 1000 transactions:")
print((commodity_counts < 1000).sum())

Category count summary:
count       306.000000
mean       8385.931373
std       14051.194735
min           1.000000
25%         575.500000
50%        2993.500000
75%       10561.250000
max      117532.000000
Name: count, dtype: float64

Categories with fewer than 100 transactions:
30

Categories with fewer than 500 transactions:
72

Categories with fewer than 1000 transactions:
96


#### Aggregate Purchasing Data by Household and Commodity

Group transactions by household, commodity category and discount status. Calculate the number of multi-unit and single-unit purchases in each group. This aggregation will allow us to examine differences across all commodity categories while accounting for repeated observations from the same household.

In [28]:
rq4_household_counts = (
    rq4_data
    .groupby(
        [
            "household_key",
            "COMMODITY_DESC",
            "discount_status",
            "purchase_type"
        ]
    )
    .size()
    .unstack("purchase_type", fill_value=0)
    .reset_index()
)

rq4_household_counts["total_purchases"] = (
    rq4_household_counts["Multi-Unit"]
    + rq4_household_counts["Single-Unit"]
)

rq4_household_counts["multi_unit_rate"] = (
    rq4_household_counts["Multi-Unit"]
    / rq4_household_counts["total_purchases"]
)

print("Shape:", rq4_household_counts.shape)
rq4_household_counts.head()

Shape: (427014, 7)


purchase_type,household_key,COMMODITY_DESC,discount_status,Multi-Unit,Single-Unit,total_purchases,multi_unit_rate
0,1,,Discounted,0,7,7,0.000000
1,1,,Non-Discounted,0,6,6,0.000000
2,1,AIR CARE,Discounted,4,6,10,0.400000
3,1,AIR CARE,Non-Discounted,1,6,7,0.142857
4,1,ANALGESICS,Discounted,0,1,1,0.000000


#### Examine Household and Transaction Coverage by Commodity

Calculate the number of households and total purchases represented in each commodity category. This will help us identify categories with limited data and assess whether the category-level statistical comparisons are reliable.

In [29]:
rq4_category_coverage = (
    rq4_data
    .groupby("COMMODITY_DESC")
    .agg(
        total_purchases=("purchase_type", "size"),
        households=("household_key", "nunique")
    )
    .sort_values("total_purchases")
)

print("Category coverage summary:")
print(rq4_category_coverage.describe())

print("\nCategories with the fewest purchases:")
print(rq4_category_coverage.head(15))

print("\nCategories with the fewest households:")
print(rq4_category_coverage.sort_values("households").head(15))

Category coverage summary:
       total_purchases   households
count       306.000000   306.000000
mean       8385.931373   935.813725
std       14051.194735   709.143363
min           1.000000     1.000000
25%         575.500000   284.750000
50%        2993.500000   830.500000
75%       10561.250000  1561.750000
max      117532.000000  2421.000000

Categories with the fewest purchases:
                              total_purchases  households
COMMODITY_DESC                                           
MISCELLANEOUS CROUTONS                      1           1
FROZEN PACKAGE MEAT                         1           1
EASTER LILY                                 1           1
MISCELLANEOUS(CORP USE ONLY)                1           1
NDAIRY/TEAS/JUICE/SOD                       2           2
TOYS                                        5           4
PKG.SEAFOOD MISC                            6           2
BULK FOODS                                  9           8
HOME HEALTH CARE              

#### Verify Commodity Category Coverage

Verify the number of distinct commodity categories in the current dataset to ensure that all categories are included in the analysis. This check will help confirm that no categories were unintentionally excluded during preprocessing.

In [30]:
print("Number of categories:", rq4_data["COMMODITY_DESC"].nunique())
print("Dataset shape:", rq4_data.shape)

print("\nCategory counts:")
print(rq4_data["COMMODITY_DESC"].value_counts().tail(10))

Number of categories: 306
Dataset shape: (2566095, 27)

Category counts:
COMMODITY_DESC
BOUQUET (NON ROSE)              10
BULK FOODS                       9
HOME HEALTH CARE                 9
PKG.SEAFOOD MISC                 6
TOYS                             5
NDAIRY/TEAS/JUICE/SOD            2
EASTER LILY                      1
MISCELLANEOUS CROUTONS           1
MISCELLANEOUS(CORP USE ONLY)     1
FROZEN PACKAGE MEAT              1
Name: count, dtype: int64


#### Check Missing Commodity Categories

Check whether any transactions or aggregated records have missing commodity categories. This is important because missing categories should not be treated as a separate commodity when analysing category-level differences.

In [31]:
print("Missing commodity categories in transaction data:")
print(rq4_data["COMMODITY_DESC"].isna().sum())

print("\nMissing commodity categories in aggregated data:")
print(rq4_household_counts["COMMODITY_DESC"].isna().sum())

print("\nRows with missing commodity categories:")
rq4_household_counts[
    rq4_household_counts["COMMODITY_DESC"].isna()
].head()

Missing commodity categories in transaction data:
0

Missing commodity categories in aggregated data:
0

Rows with missing commodity categories:


purchase_type,household_key,COMMODITY_DESC,discount_status,Multi-Unit,Single-Unit,total_purchases,multi_unit_rate


#### Examine Discount-Group Coverage Across Categories

Check how many households and purchases are available in each discount group for every commodity category. This helps identify categories where the comparison between discounted and non-discounted purchases may be unreliable.

In [32]:
rq4_discount_coverage = (
    rq4_data
    .groupby(["COMMODITY_DESC", "discount_status"])
    .agg(
        total_purchases=("purchase_type", "size"),
        households=("household_key", "nunique")
    )
    .unstack("discount_status", fill_value=0)
)

print("Discount-group coverage shape:", rq4_discount_coverage.shape)

print("\nCategories missing one or both discount groups:")
print(
    (
        (rq4_discount_coverage["households"]["Discounted"] == 0)
        | (rq4_discount_coverage["households"]["Non-Discounted"] == 0)
    ).sum()
)

rq4_discount_coverage.head()

Discount-group coverage shape: (306, 4)

Categories missing one or both discount groups:
26


total_purchases                households               
discount_status         Discounted Non-Discounted Discounted Non-Discounted
COMMODITY_DESC                                                             
                              4945           2894       1169            930
(CORP USE ONLY)                  0             61          0             54
ADULT INCONTINENCE             303            250         66            100
AIR CARE                      3165           2391        854            785
ANALGESICS                    1733           3036        794           1099

#### Identify Categories Missing a Discount Group

List the categories that have purchases in only one discount group. These categories cannot provide a direct discounted versus non-discounted comparison.

In [33]:
missing_discount_categories = rq4_discount_coverage[
    (rq4_discount_coverage["households"]["Discounted"] == 0)
    | (rq4_discount_coverage["households"]["Non-Discounted"] == 0)
]

print("Number of affected categories:", len(missing_discount_categories))

print("\nCategories missing a discount group:")
display(missing_discount_categories)

Number of affected categories: 26

Categories missing a discount group:


total_purchases                households  \
discount_status                     Discounted Non-Discounted Discounted   
COMMODITY_DESC                                                             
(CORP USE ONLY)                              0             61          0   
BOTTLE DEPOSITS                              0             95          0   
BOUQUET (NON ROSE)                           0             10          0   
BULK FOODS                                   0              9          0   
CIGARS                                       0            167          0   
COFFEE SHOP SWEET GOODS&RETAIL               0             84          0   
COUPON                                       0             70          0   
DELI SUPPLIES                                0            594          0   
EASTER LILY                                  0              1          0   
FROZEN PACKAGE MEAT                          0              1          0   
FRZN ICE                                     0           2469          0   
GIFT & FRUIT BASKETS                         0             21          0   
GLASSES/VISION AIDS                          0            139          0   
HOME HEALTH CARE                             0              9          0   
MEAT SUPPLIES                                0             44          0   
MISCELLANEOUS CROUTONS                       0              1          0   
MISCELLANEOUS HBC                            0             28          0   
MISCELLANEOUS(CORP USE ONLY)                 0              1          0   
NDAIRY/TEAS/JUICE/SOD                        0              2          0   
PHARMACY                                     0             13          0   
PKG.SEAFOOD MISC                             0              6          0   
PROD SUPPLIES                                0             37          0   
PROPANE                                      0             57          0   
QUICK SERVICE                                0            133          0   
TICKETS                                      0            715          0   
VEAL                                         0            106          0   

                                               
discount_status                Non-Discounted  
COMMODITY_DESC                                 
(CORP USE ONLY)                            54  
BOTTLE DEPOSITS                            18  
BOUQUET (NON ROSE)                         10  
BULK FOODS                                  8  
CIGARS                                     73  
COFFEE SHOP SWEET GOODS&RETAIL             51  
COUPON                                     65  
DELI SUPPLIES                             420  
EASTER LILY                                 1  
FROZEN PACKAGE MEAT                         1  
FRZN ICE                                  871  
GIFT & FRUIT BASKETS                       20  
GLASSES/VISION AIDS                        96  
HOME HEALTH CARE                            8  
MEAT SUPPLIES                              36  
MISCELLANEOUS CROUTONS                      1  
MISCELLANEOUS HBC                          22  
MISCELLANEOUS(CORP USE ONLY)                1  
NDAIRY/TEAS/JUICE/SOD                       2  
PHARMACY                                   13  
PKG.SEAFOOD MISC                            2  
PROD SUPPLIES                              26  
PROPANE                                    44  
QUICK SERVICE                              60  
TICKETS                                   324  
VEAL                                       65

#### Select Categories with Both Discount Groups

For the statistical comparison, retain categories that have both discounted and non-discounted purchases. Categories without either group cannot provide a meaningful comparison, so they will remain in the descriptive data but not in the interaction model.

In [34]:
eligible_categories = rq4_discount_coverage[
    (rq4_discount_coverage["total_purchases"]["Discounted"] > 0)
    & (rq4_discount_coverage["total_purchases"]["Non-Discounted"] > 0)
].index

rq4_model_data = rq4_household_counts[
    rq4_household_counts["COMMODITY_DESC"].isin(eligible_categories)
].copy()

print("Eligible categories:", len(eligible_categories))
print("Model dataset shape:", rq4_model_data.shape)
print(
    "Unique categories in model data:",
    rq4_model_data["COMMODITY_DESC"].nunique()
)

Eligible categories: 280
Model dataset shape: (424722, 7)
Unique categories in model data: 280


#### Examine Sample Sizes of Eligible Categories

Inspect the transaction and household counts across the 280 eligible categories. This helps us assess whether the category-level comparisons have sufficient data and whether some estimates may be unstable.

In [35]:
rq4_model_coverage = (
    rq4_model_data
    .groupby("COMMODITY_DESC")
    .agg(
        total_purchases=("total_purchases", "sum"),
        households=("household_key", "nunique"),
        household_discount_groups=("discount_status", "count")
    )
    .sort_values("total_purchases")
)

print("Eligible category coverage summary:")
display(rq4_model_coverage.describe())

print("\n10 categories with the fewest purchases:")
display(rq4_model_coverage.head(10))

print("\n10 categories with the fewest households:")
display(rq4_model_coverage.sort_values("households").head(10))

Eligible category coverage summary:


,total_purchases,households,household_discount_groups
count,280.000000,280.00000,280.000000
mean,9147.221429,1014.52500,1516.864286
std,14455.712257,688.06432,1201.602727
min,5.000000,4.00000,5.000000
25%,987.500000,373.75000,459.500000
50%,4022.500000,977.50000,1273.500000
75%,11448.250000,1601.00000,2402.500000
max,117532.000000,2421.00000,4585.000000



10 categories with the fewest purchases:


,total_purchases,households,household_discount_groups
COMMODITY_DESC,,,
TOYS,5,4,5
RW FRESH PROCESSED MEAT,28,17,17
NATURAL VITAMINS,32,27,29
WATCHES/CALCULATORS/LOBBY,43,36,39
NATURAL HBC,49,29,29
RESTRICTED DIET,51,26,30
LONG DISTANCE CALLING CARDS,61,30,34
FRZN SEAFOOD,64,49,53
SMOKING CESSATIONS,79,28,33



10 categories with the fewest households:


,total_purchases,households,household_discount_groups
COMMODITY_DESC,,,
TOYS,5,4,5
RW FRESH PROCESSED MEAT,28,17,17
RESTRICTED DIET,51,26,30
NATURAL VITAMINS,32,27,29
SMOKING CESSATIONS,79,28,33
NON EDIBLE PRODUCTS,95,28,39
NATURAL HBC,49,29,29
LONG DISTANCE CALLING CARDS,61,30,34
BABYFOOD,475,33,49


#### Compare Discounted and Non-Discounted Sample Sizes

Examine the number of purchases and households in each discount group across eligible categories. This helps identify categories where one group has very limited data, which could make category-specific estimates unstable.

In [36]:
rq4_group_coverage = (
    rq4_model_data
    .groupby(["COMMODITY_DESC", "discount_status"])
    .agg(
        purchases=("total_purchases", "sum"),
        households=("household_key", "nunique")
    )
    .unstack("discount_status", fill_value=0)
)

print("Discounted purchases summary:")
display(rq4_group_coverage["purchases"]["Discounted"].describe())

print("\nNon-discounted purchases summary:")
display(rq4_group_coverage["purchases"]["Non-Discounted"].describe())

print("\nCategories with the fewest discounted purchases:")
display(
    rq4_group_coverage
    .sort_values(("purchases", "Discounted"))
    .head(15)
)

print("\nCategories with the fewest non-discounted purchases:")
display(
    rq4_group_coverage
    .sort_values(("purchases", "Non-Discounted"))
    .head(15)
)

Discounted purchases summary:


count      280.000000
mean      4629.432143
std       9115.913885
min          1.000000
25%        147.250000
50%       1191.000000
75%       5302.000000
max      76317.000000
Name: Discounted, dtype: float64


Non-discounted purchases summary:


count      280.000000
mean      4517.789286
std       6281.422875
min          4.000000
25%        641.750000
50%       2238.000000
75%       5522.750000
max      43223.000000
Name: Non-Discounted, dtype: float64


Categories with the fewest discounted purchases:


purchases                households  \
discount_status             Discounted Non-Discounted Discounted   
COMMODITY_DESC                                                     
FIREWORKS                            1            217          1   
TOYS                                 1              4          1   
FLORAL-FOLIAGE PLANTS                1            283          1   
FLORAL BALLOONS                      2            404          2   
SUSHI                                2            851          1   
BAKERY PARTY TRAYS                   2            136          2   
FLORAL-ACCESSORIES                   2             82          2   
RW FRESH PROCESSED MEAT              2             26          2   
NATURAL HBC                          5             44          4   
OVERNIGHT PHOTOFINISHING             5            170          4   
LONG DISTANCE CALLING CARDS          6             55          5   
FLORAL- HARD GOODS                   6            252          5   
MISC WINE                            7           1434          4   
J-HOOKS                              8            950          8   
WATCHES/CALCULATORS/LOBBY            9             34          9   

                                            
discount_status             Non-Discounted  
COMMODITY_DESC                              
FIREWORKS                              132  
TOYS                                     4  
FLORAL-FOLIAGE PLANTS                  188  
FLORAL BALLOONS                        259  
SUSHI                                  125  
BAKERY PARTY TRAYS                      96  
FLORAL-ACCESSORIES                      65  
RW FRESH PROCESSED MEAT                 15  
NATURAL HBC                             25  
OVERNIGHT PHOTOFINISHING                92  
LONG DISTANCE CALLING CARDS             29  
FLORAL- HARD GOODS                     165  
MISC WINE                              591  
J-HOOKS                                541  
WATCHES/CALCULATORS/LOBBY               30


Categories with the fewest non-discounted purchases:


purchases                households  \
discount_status              Discounted Non-Discounted Discounted   
COMMODITY_DESC                                                      
TOYS                                  1              4          1   
NATURAL VITAMINS                      9             23          9   
RW FRESH PROCESSED MEAT               2             26          2   
WATCHES/CALCULATORS/LOBBY             9             34          9   
RESTRICTED DIET                      14             37         10   
FRZN SEAFOOD                         24             40         21   
NATURAL HBC                           5             44          4   
LONG DISTANCE CALLING CARDS           6             55          5   
SMOKING CESSATIONS                   18             61          8   
NON EDIBLE PRODUCTS                  24             71         13   
FLORAL-ACCESSORIES                    2             82          2   
DRY TEA/COFFEE/COCO MIX             114             84         51   
HOME FURNISHINGS                     21             85         12   
PORTABLE ELECTRIC APPLIANCES         40             89         40   
EXOTIC GAME/FOWL                    106             98         80   

                                             
discount_status              Non-Discounted  
COMMODITY_DESC                               
TOYS                                      4  
NATURAL VITAMINS                         20  
RW FRESH PROCESSED MEAT                  15  
WATCHES/CALCULATORS/LOBBY                30  
RESTRICTED DIET                          20  
FRZN SEAFOOD                             32  
NATURAL HBC                              25  
LONG DISTANCE CALLING CARDS              29  
SMOKING CESSATIONS                       25  
NON EDIBLE PRODUCTS                      26  
FLORAL-ACCESSORIES                       65  
DRY TEA/COFFEE/COCO MIX                  49  
HOME FURNISHINGS                         66  
PORTABLE ELECTRIC APPLIANCES             80  
EXOTIC GAME/FOWL                         71

#### Inspect Available Commodity Groupings

Check whether the dataset contains a broader department-level classification. A broader grouping may allow us to examine differences in the discount–multi-unit relationship with less sparse data.

In [37]:
print("Columns related to product classification:")

print([
    col for col in rq4_data.columns
    if any(term in col.lower() for term in ["department", "commodity", "category"])
])

if "DEPARTMENT" in rq4_data.columns:
    print("\nNumber of departments:", rq4_data["DEPARTMENT"].nunique())
    print("\nTransactions by department:")
    print(rq4_data["DEPARTMENT"].value_counts())

Columns related to product classification:
['COMMODITY_DESC', 'SUB_COMMODITY_DESC']


#### Analyze Discount Associations Across Commodity Categories

Fit separate household-clustered Generalized Estimating Equation (GEE) models for the 280 eligible commodity categories. Each model estimates the association between discount status and multi-unit purchasing within one category, accounting for repeated observations from the same household. Extract the discount coefficient, odds ratio, confidence interval and p-value for each category.

In [38]:
# Store results for each commodity category
rq4_category_results = []

# Fit a separate GEE model for each category
for category, group in rq4_model_data.groupby("COMMODITY_DESC"):

    try:
        # Dependent variable
        y = group["multi_unit_rate"].to_numpy()

        # Independent variable: 0 = Non-Discounted, 1 = Discounted
        discount = (
            group["discount_status"] == "Discounted"
        ).astype(int).to_numpy()

        # Create design matrix with intercept and discount status
        X = np.column_stack([
            np.ones(len(group)),
            discount
        ])

        # Fit household-clustered GEE
        model = sm.GEE(
            endog=y,
            exog=X,
            groups=group["household_key"].to_numpy(),
            weights=group["total_purchases"].to_numpy(),
            family=sm.families.Binomial(),
            cov_struct=sm.cov_struct.Independence()
        )

        result = model.fit(maxiter=100)

        # Extract coefficients using positional indexing
        params = np.asarray(result.params)
        standard_errors = np.asarray(result.bse)
        p_values = np.asarray(result.pvalues)

        coef = params[1]
        se = standard_errors[1]
        p_value = p_values[1]

        rq4_category_results.append({
            "COMMODITY_DESC": category,
            "coefficient": coef,
            "standard_error": se,
            "odds_ratio": np.exp(coef),
            "ci_lower": np.exp(coef - 1.96 * se),
            "ci_upper": np.exp(coef + 1.96 * se),
            "p_value": p_value,
            "converged": result.converged
        })

    except Exception as error:
        print(f"Could not fit {category}: {error}")

# Convert results into a DataFrame
rq4_category_results = pd.DataFrame(rq4_category_results)

print("Successfully fitted categories:", len(rq4_category_results))
print("Categories not fitted:", 280 - len(rq4_category_results))

display(rq4_category_results.head())

/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/home/rohan/GitHub/effect-of-discount-on-customer-purchase/.venv/lib/python3.14/site-pac

Successfully fitted categories: 280
Categories not fitted: 0


,COMMODITY_DESC,coefficient,standard_error,odds_ratio,ci_lower,ci_upper,p_value,converged
0,,-6.298641e-15,0.043039,1.000000,0.919103,1.088017,1.000000e+00,True
1,ADULT INCONTINENCE,6.941697e-01,0.436666,2.002046,0.850699,4.711639,1.119018e-01,True
2,AIR CARE,4.047235e-01,0.087495,1.498888,1.262676,1.779289,3.733775e-06,True
3,ANALGESICS,3.054152e+00,0.194923,21.203197,14.470374,31.068688,2.482671e-55,True
4,ANTACIDS,2.945675e+00,0.292505,19.023490,10.722724,33.750114,7.458356e-24,True


#### Check Model Convergence and Numerical Stability

Check whether all category-level models converged and whether their coefficients, standard errors, odds ratios, confidence intervals and p-values are finite and valid. Identify estimates that may be unreliable because of perfect separation or numerical problems.

In [39]:
# Check convergence
print("Convergence status:")
print(rq4_category_results["converged"].value_counts(dropna=False))

# Check for invalid numerical estimates
numeric_columns = [
    "coefficient", "standard_error", "odds_ratio",
    "ci_lower", "ci_upper", "p_value"
]

finite_estimates = np.isfinite(
    rq4_category_results[numeric_columns]
).all(axis=1)

valid_estimates = (
    finite_estimates
    & (rq4_category_results["standard_error"] > 0)
    & (rq4_category_results["p_value"].between(0, 1))
    & (rq4_category_results["ci_lower"] > 0)
    & (rq4_category_results["ci_upper"] > 0)
)

print("\nTotal category models:", len(rq4_category_results))
print("Numerically valid estimates:", valid_estimates.sum())
print("Potentially unreliable estimates:", (~valid_estimates).sum())

print("\nCategories with potentially unreliable estimates:")
display(
    rq4_category_results.loc[
        ~valid_estimates,
        ["COMMODITY_DESC"] + numeric_columns + ["converged"]
    ]
)

# Inspect unusually large odds ratios
print("\nLargest odds ratios:")
display(
    rq4_category_results
    .sort_values("odds_ratio", ascending=False)
    .head(10)
)

Convergence status:
converged
True    280
Name: count, dtype: int64

Total category models: 280
Numerically valid estimates: 280
Potentially unreliable estimates: 0

Categories with potentially unreliable estimates:


,COMMODITY_DESC,coefficient,standard_error,odds_ratio,ci_lower,ci_upper,p_value,converged



Largest odds ratios:


,COMMODITY_DESC,coefficient,standard_error,odds_ratio,ci_lower,ci_upper,p_value,converged
263,TOYS,47.132137,1.322874,2.945961e+20,2.203808e+19,3.938039e+21,5.056751e-278,True
183,NATURAL VITAMINS,24.313305,0.807573,3.623550e+10,7.442414e+09,1.764228e+11,3.969462e-199,True
110,FLORAL-FLOWERING PLANTS,3.674491,0.658370,3.942857e+01,1.084915e+01,1.432935e+02,2.388679e-08,True
274,VITAMINS,3.423127,0.210569,3.066516e+01,2.029576e+01,4.633244e+01,2.006709e-59,True
69,CORN,3.294704,0.342632,2.696944e+01,1.377899e+01,5.278693e+01,6.853343e-22,True
236,SEWING,3.102791,0.740015,2.226000e+01,5.219298e+00,9.493760e+01,2.754366e-05,True
3,ANALGESICS,3.054152,0.194923,2.120320e+01,1.447037e+01,3.106869e+01,2.482671e-55,True
4,ANTACIDS,2.945675,0.292505,1.902349e+01,1.072272e+01,3.375011e+01,7.458356e-24,True
145,HOME FREEZING & CANNING SUPPLY,2.081843,0.876958,8.019231e+00,1.437643e+00,4.473159e+01,1.759957e-02,True
133,GARDEN CENTER,2.061395,0.321964,7.856925e+00,4.180143e+00,1.476774e+01,1.527941e-10,True


#### Assess the Number of Household Clusters per Category

Examine how many households contribute observations to each commodity category. GEE uses household-clustered robust standard errors, so categories with very few independent households may have unreliable statistical inference even when their models converge.

In [40]:
# Summarize household cluster counts
print("Household coverage across eligible categories:")
display(rq4_model_coverage["households"].describe())

# Count categories by household coverage
print("\nCategories with fewer than 10 households:")
print((rq4_model_coverage["households"] < 10).sum())

print("\nCategories with fewer than 30 households:")
print((rq4_model_coverage["households"] < 30).sum())

print("\nCategories with fewer than 50 households:")
print((rq4_model_coverage["households"] < 50).sum())

# Show categories with the fewest household clusters
display(rq4_model_coverage.sort_values("households").head(20))

Household coverage across eligible categories:


count     280.00000
mean     1014.52500
std       688.06432
min         4.00000
25%       373.75000
50%       977.50000
75%      1601.00000
max      2421.00000
Name: households, dtype: float64


Categories with fewer than 10 households:
1

Categories with fewer than 30 households:
7

Categories with fewer than 50 households:
11


,total_purchases,households,household_discount_groups
COMMODITY_DESC,,,
TOYS,5,4,5
RW FRESH PROCESSED MEAT,28,17,17
RESTRICTED DIET,51,26,30
NATURAL VITAMINS,32,27,29
SMOKING CESSATIONS,79,28,33
NON EDIBLE PRODUCTS,95,28,39
NATURAL HBC,49,29,29
LONG DISTANCE CALLING CARDS,61,30,34
BABYFOOD,475,33,49


#### Select Reliable Category-Level Estimates

Retain categories with at least 30 household clusters for the primary inferential analysis. This reduces dependence on unreliable large-sample standard errors from very small clusters. Exclude estimates with invalid numerical results. Preserve all valid estimates separately for a sensitivity analysis to assess whether including smaller categories changes the findings.

In [41]:
# Add household counts to the category-level model results
rq4_results_checked = rq4_category_results.merge(
    rq4_model_coverage[["households"]].reset_index(),
    on="COMMODITY_DESC",
    how="left"
)

# Identify numerically valid estimates
rq4_results_checked["valid_estimate"] = (
    np.isfinite(rq4_results_checked[
        ["coefficient", "standard_error", "odds_ratio",
         "ci_lower", "ci_upper", "p_value"]
    ]).all(axis=1)
    & (rq4_results_checked["standard_error"] > 0)
    & rq4_results_checked["p_value"].between(0, 1)
    & (rq4_results_checked["ci_lower"] > 0)
)

# Primary analysis: at least 30 household clusters
rq4_primary_results = rq4_results_checked[
    (rq4_results_checked["households"] >= 30)
    & rq4_results_checked["valid_estimate"]
].copy()

# Sensitivity analysis: all numerically valid estimates
rq4_sensitivity_results = rq4_results_checked[
    rq4_results_checked["valid_estimate"]
].copy()

print("Total categories:", len(rq4_results_checked))
print("Primary analysis categories:", len(rq4_primary_results))
print("Sensitivity analysis categories:", len(rq4_sensitivity_results))

print("\nCategories excluded from primary analysis:")
display(
    rq4_results_checked[
        (rq4_results_checked["households"] < 30)
        | (~rq4_results_checked["valid_estimate"])
    ][["COMMODITY_DESC", "households", "valid_estimate"]]
)

Total categories: 280
Primary analysis categories: 273
Sensitivity analysis categories: 280

Categories excluded from primary analysis:


,COMMODITY_DESC,households,valid_estimate
182,NATURAL HBC,29,True
183,NATURAL VITAMINS,27,True
187,NON EDIBLE PRODUCTS,28,True
220,RESTRICTED DIET,26,True
224,RW FRESH PROCESSED MEAT,17,True
242,SMOKING CESSATIONS,28,True
263,TOYS,4,True


#### Apply Holm Correction for Multiple Comparisons

Test the discount association within each commodity category and adjust the p-values using the Holm method. Since we conduct multiple category-level tests, this correction controls the family-wise error rate and reduces the risk of false-positive findings. Use the 273 categories selected for the primary analysis.

In [42]:
# Import multiple-testing correction
from statsmodels.stats.multitest import multipletests

# Apply Holm correction
holm_reject, holm_pvalues, _, _ = multipletests(
    rq4_primary_results["p_value"],
    alpha=0.05,
    method="holm"
)

# Store adjusted results
rq4_primary_results["holm_p_value"] = holm_pvalues
rq4_primary_results["significant_after_holm"] = holm_reject

# Sort by adjusted p-value
rq4_primary_results = rq4_primary_results.sort_values(
    "holm_p_value"
).reset_index(drop=True)

# Summarize findings
print("Primary analysis categories:", len(rq4_primary_results))

print(
    "Significant before correction:",
    (rq4_primary_results["p_value"] < 0.05).sum()
)

print(
    "Significant after Holm correction:",
    rq4_primary_results["significant_after_holm"].sum()
)

print("\nTop 15 category results:")
display(
    rq4_primary_results[
        [
            "COMMODITY_DESC",
            "households",
            "odds_ratio",
            "ci_lower",
            "ci_upper",
            "p_value",
            "holm_p_value",
            "significant_after_holm"
        ]
    ].head(15)
)

Primary analysis categories: 273
Significant before correction: 179
Significant after Holm correction: 138

Top 15 category results:


,COMMODITY_DESC,households,odds_ratio,ci_lower,ci_upper,p_value,holm_p_value,significant_after_holm
0,SHOE CARE,181,8.114090e-10,3.121751e-10,2.109024e-09,0.000000e+00,0.000000e+00,True
1,FLORAL- HARD GOODS,169,4.311399e-10,1.565326e-10,1.187494e-09,0.000000e+00,0.000000e+00,True
2,OVERNIGHT PHOTOFINISHING,96,4.399248e-10,1.452332e-10,1.332573e-09,0.000000e+00,0.000000e+00,True
3,MAGAZINE,1211,1.253057e-09,5.999447e-10,2.617159e-09,0.000000e+00,0.000000e+00,True
4,BOOKSTORE,639,2.718782e-10,1.546411e-10,4.779958e-10,0.000000e+00,0.000000e+00,True
5,PERSONAL CARE APPLIANCES,138,3.679631e-10,1.094980e-10,1.236523e-09,2.568517e-270,6.883626e-268,True
6,FLORAL BALLOONS,260,1.072063e-09,2.612743e-10,4.398898e-09,8.026981e-181,2.143204e-178,True
7,FLORAL-ACCESSORIES,66,6.055436e-10,1.331396e-10,2.754125e-09,4.703479e-166,1.251125e-163,True
8,BAKERY PARTY TRAYS,98,3.228767e-09,7.233270e-10,1.441248e-08,1.031418e-144,2.733256e-142,True
9,FLUID MILK PRODUCTS,2421,4.090432e+00,3.671230e+00,4.557500e+00,8.028359e-144,2.119487e-141,True


#### Investigate Extreme Category-Level Odds Ratios

Examine the underlying multi-unit and single-unit purchase counts in both discount groups. This helps identify categories where one outcome is absent or nearly absent in a discount group, which can cause perfect or quasi-perfect separation and unreliable odds-ratio estimates.

In [43]:
# Aggregate actual purchase counts by category and discount status
rq4_outcome_counts = (
    rq4_model_data
    .groupby(["COMMODITY_DESC", "discount_status"])
    .agg(
        multi_unit=("Multi-Unit", "sum"),
        single_unit=("Single-Unit", "sum"),
        households=("household_key", "nunique")
    )
    .reset_index()
)

# Calculate multi-unit purchasing rate
rq4_outcome_counts["multi_unit_rate"] = (
    rq4_outcome_counts["multi_unit"]
    / (rq4_outcome_counts["multi_unit"] + rq4_outcome_counts["single_unit"])
)

# Reshape for comparison between discount groups
rq4_outcome_comparison = rq4_outcome_counts.pivot(
    index="COMMODITY_DESC",
    columns="discount_status",
    values=["multi_unit", "single_unit", "households", "multi_unit_rate"]
)

# Identify categories with zero multi-unit or single-unit purchases
rq4_outcome_comparison["zero_multi_unit_group"] = (
    (rq4_outcome_comparison["multi_unit"]["Discounted"] == 0)
    | (rq4_outcome_comparison["multi_unit"]["Non-Discounted"] == 0)
)

rq4_outcome_comparison["zero_single_unit_group"] = (
    (rq4_outcome_comparison["single_unit"]["Discounted"] == 0)
    | (rq4_outcome_comparison["single_unit"]["Non-Discounted"] == 0)
)

print("Categories with zero multi-unit purchases in a group:")
print(rq4_outcome_comparison["zero_multi_unit_group"].sum())

print("\nCategories with zero single-unit purchases in a group:")
print(rq4_outcome_comparison["zero_single_unit_group"].sum())

# Inspect the most extreme odds ratios
extreme_categories = rq4_primary_results[
    (rq4_primary_results["odds_ratio"] < 0.01)
    | (rq4_primary_results["odds_ratio"] > 100)
]["COMMODITY_DESC"].tolist()

print("\nExtreme odds-ratio categories:", len(extreme_categories))

display(
    rq4_outcome_comparison.loc[
        rq4_outcome_comparison.index.isin(extreme_categories)
    ]
)

Categories with zero multi-unit purchases in a group:
19

Categories with zero single-unit purchases in a group:
1

Extreme odds-ratio categories: 14


multi_unit                single_unit  \
discount_status             Discounted Non-Discounted  Discounted   
COMMODITY_DESC                                                      
BAKERY PARTY TRAYS                 0.0           16.0         2.0   
BOOKSTORE                          0.0           52.0        30.0   
FIREWORKS                          0.0           54.0         1.0   
FLORAL BALLOONS                    0.0           52.0         2.0   
FLORAL- HARD GOODS                 0.0           30.0         6.0   
FLORAL-ACCESSORIES                 0.0           17.0         2.0   
FLORAL-FOLIAGE PLANTS              0.0           40.0         1.0   
FRAGRANCES                         0.0            1.0        10.0   
LONG DISTANCE CALLING CARDS        0.0            2.0         6.0   
MAGAZINE                           0.0           61.0        20.0   
OVERNIGHT PHOTOFINISHING           0.0           45.0         5.0   
PERSONAL CARE APPLIANCES           0.0            3.0        30.0   
SHOE CARE                          0.0            7.0        10.0   
SUSHI                              0.0           52.0         2.0   

                                           households                 \
discount_status             Non-Discounted Discounted Non-Discounted   
COMMODITY_DESC                                                         
BAKERY PARTY TRAYS                   120.0        2.0           96.0   
BOOKSTORE                           1793.0       26.0          630.0   
FIREWORKS                            163.0        1.0          132.0   
FLORAL BALLOONS                      352.0        2.0          259.0   
FLORAL- HARD GOODS                   222.0        5.0          165.0   
FLORAL-ACCESSORIES                    65.0        2.0           65.0   
FLORAL-FOLIAGE PLANTS                243.0        1.0          188.0   
FRAGRANCES                           108.0        9.0           87.0   
LONG DISTANCE CALLING CARDS           53.0        5.0           29.0   
MAGAZINE                            9694.0       14.0         1210.0   
OVERNIGHT PHOTOFINISHING             125.0        4.0           92.0   
PERSONAL CARE APPLIANCES             140.0       30.0          118.0   
SHOE CARE                            265.0       10.0          177.0   
SUSHI                                799.0        1.0          125.0   

                            multi_unit_rate                 \
discount_status                  Discounted Non-Discounted   
COMMODITY_DESC                                               
BAKERY PARTY TRAYS                      0.0       0.117647   
BOOKSTORE                               0.0       0.028184   
FIREWORKS                               0.0       0.248848   
FLORAL BALLOONS                         0.0       0.128713   
FLORAL- HARD GOODS                      0.0       0.119048   
FLORAL-ACCESSORIES                      0.0       0.207317   
FLORAL-FOLIAGE PLANTS                   0.0       0.141343   
FRAGRANCES                              0.0       0.009174   
LONG DISTANCE CALLING CARDS             0.0       0.036364   
MAGAZINE                                0.0       0.006253   
OVERNIGHT PHOTOFINISHING                0.0       0.264706   
PERSONAL CARE APPLIANCES                0.0       0.020979   
SHOE CARE                               0.0       0.025735   
SUSHI                                   0.0       0.061105   

                            zero_multi_unit_group zero_single_unit_group  
discount_status                                                           
COMMODITY_DESC                                                            
BAKERY PARTY TRAYS                           True                  False  
BOOKSTORE                                    True                  False  
FIREWORKS                                    True                  False  
FLORAL BALLOONS                              True                  False  
FLORAL- HARD GOODS                           True

#### Identify Categories with Complete Outcome Data

For a reliable category-level odds-ratio estimate, both multi-unit and single-unit purchases must be observed in both discount groups. Identify categories with zero counts in any of these four groups. Retain them for descriptive analysis but exclude them from the primary inferential analysis because their discount effects may be affected by separation.

In [44]:
# Identify categories with non-zero counts in all four groups
outcome_support = (
    (rq4_outcome_comparison["multi_unit"]["Discounted"] > 0)
    & (rq4_outcome_comparison["multi_unit"]["Non-Discounted"] > 0)
    & (rq4_outcome_comparison["single_unit"]["Discounted"] > 0)
    & (rq4_outcome_comparison["single_unit"]["Non-Discounted"] > 0)
)

# List categories with empty outcome groups
categories_with_separation = outcome_support[
    ~outcome_support
].index.tolist()

print("Categories with an empty outcome group:",
      len(categories_with_separation))

print("\nCategories excluded due to empty outcome groups:")
print(categories_with_separation)

# Rebuild the primary analysis dataset
rq4_primary_results = rq4_results_checked[
    (rq4_results_checked["households"] >= 30)
    & rq4_results_checked["valid_estimate"]
    & rq4_results_checked["COMMODITY_DESC"].isin(
        outcome_support[outcome_support].index
    )
].copy()

print("\nCategories retained for primary analysis:",
      len(rq4_primary_results))

print("Categories excluded from primary analysis:",
      280 - len(rq4_primary_results))

Categories with an empty outcome group: 19

Categories excluded due to empty outcome groups:
[' ', 'BAKERY PARTY TRAYS', 'BOOKSTORE', 'FIREWORKS', 'FLORAL BALLOONS', 'FLORAL- HARD GOODS', 'FLORAL-ACCESSORIES', 'FLORAL-FOLIAGE PLANTS', 'FRAGRANCES', 'LONG DISTANCE CALLING CARDS', 'MAGAZINE', 'NATURAL HBC', 'NATURAL VITAMINS', 'OVERNIGHT PHOTOFINISHING', 'PERSONAL CARE APPLIANCES', 'RW FRESH PROCESSED MEAT', 'SHOE CARE', 'SUSHI', 'TOYS']

Categories retained for primary analysis: 258
Categories excluded from primary analysis: 22


#### Apply Final Holm Correction

Apply the Holm multiple-testing correction to the 258 categories retained for the primary analysis. This controls the family-wise error rate while accounting for the multiple category-level tests. Summarize the number of statistically significant associations after correction.

In [45]:
# Apply Holm correction to the final primary analysis
holm_reject, holm_pvalues, _, _ = multipletests(
    rq4_primary_results["p_value"],
    alpha=0.05,
    method="holm"
)

rq4_primary_results["holm_p_value"] = holm_pvalues
rq4_primary_results["significant_after_holm"] = holm_reject

# Sort results by adjusted p-value
rq4_primary_results = (
    rq4_primary_results
    .sort_values("holm_p_value")
    .reset_index(drop=True)
)

# Summarize the final results
print("Categories in primary analysis:", len(rq4_primary_results))

print(
    "Significant before correction:",
    (rq4_primary_results["p_value"] < 0.05).sum()
)

print(
    "Significant after Holm correction:",
    rq4_primary_results["significant_after_holm"].sum()
)

print("\nFinal category-level results:")
display(
    rq4_primary_results[
        [
            "COMMODITY_DESC",
            "households",
            "odds_ratio",
            "ci_lower",
            "ci_upper",
            "p_value",
            "holm_p_value",
            "significant_after_holm"
        ]
    ].head(20)
)

Categories in primary analysis: 258
Significant before correction: 165
Significant after Holm correction: 124

Final category-level results:


,COMMODITY_DESC,households,odds_ratio,ci_lower,ci_upper,p_value,holm_p_value,significant_after_holm
0,FLUID MILK PRODUCTS,2421,4.090432,3.671230,4.557500,8.028359e-144,2.071317e-141,True
1,BERRIES,1543,4.804947,4.199069,5.498245,2.553608e-115,6.562774e-113,True
2,VEGETABLES - SHELF STABLE,2138,1.796573,1.687641,1.912535,2.803834e-75,7.177815e-73,True
3,CANDY - CHECKLANE,2142,2.515399,2.274980,2.781225,2.062144e-72,5.258467e-70,True
4,CHICKEN,1891,2.633492,2.352934,2.947504,1.085276e-63,2.756602e-61,True
5,HOT DOGS,1788,4.469762,3.745530,5.334031,6.735624e-62,1.704113e-59,True
6,VITAMINS,731,30.665163,20.295765,46.332436,2.006709e-59,5.056907e-57,True
7,BACON,1694,3.709236,3.165478,4.346399,4.474835e-59,1.123184e-56,True
8,LUNCHMEAT,2147,2.199070,1.998683,2.419547,8.835699e-59,2.208925e-56,True
9,ANALGESICS,1315,21.203197,14.470374,31.068688,2.482671e-55,6.181850e-53,True


#### Summarize the Direction of Category-Level Associations

Summarize whether discounts are associated with higher or lower odds of multi-unit purchasing across commodity categories. Separate statistically significant positive associations, significant negative associations and non-significant results using the Holm-adjusted p-values.

In [46]:
# Separate significant positive and negative associations
significant_positive = rq4_primary_results[
    (rq4_primary_results["significant_after_holm"])
    & (rq4_primary_results["odds_ratio"] > 1)
]

significant_negative = rq4_primary_results[
    (rq4_primary_results["significant_after_holm"])
    & (rq4_primary_results["odds_ratio"] < 1)
]

non_significant = rq4_primary_results[
    ~rq4_primary_results["significant_after_holm"]
]

print("Total categories:", len(rq4_primary_results))

print("\nSignificant positive associations:", len(significant_positive))
print("Significant negative associations:", len(significant_negative))
print("Non-significant associations:", len(non_significant))

print("\nMedian odds ratio:", rq4_primary_results["odds_ratio"].median())

print("\nTop 10 positive associations:")
display(
    significant_positive.sort_values(
        "odds_ratio", ascending=False
    )[["COMMODITY_DESC", "households", "odds_ratio",
       "ci_lower", "ci_upper", "holm_p_value"]].head(10)
)

print("\nTop 10 negative associations:")
display(
    significant_negative.sort_values(
        "odds_ratio"
    )[["COMMODITY_DESC", "households", "odds_ratio",
       "ci_lower", "ci_upper", "holm_p_value"]].head(10)
)

Total categories: 258

Significant positive associations: 110
Significant negative associations: 14
Non-significant associations: 134

Median odds ratio: 1.5692775112804784

Top 10 positive associations:


,COMMODITY_DESC,households,odds_ratio,ci_lower,ci_upper,holm_p_value
85,FLORAL-FLOWERING PLANTS,343,39.428571,10.849146,143.293512,4.132414e-06
6,VITAMINS,731,30.665163,20.295765,46.332436,5.056907e-57
36,CORN,923,26.969442,13.778993,52.786934,1.521442e-19
107,SEWING,131,22.260000,5.219298,94.937603,4.159092e-03
9,ANALGESICS,1315,21.203197,14.470374,31.068688,6.181850e-53
34,ANTACIDS,722,19.023490,10.722724,33.750114,1.670672e-21
74,GARDEN CENTER,282,7.856925,4.180143,14.767744,2.814438e-08
91,ROSES,322,5.562171,2.910184,10.630855,3.471382e-05
113,EXOTIC GAME/FOWL,132,5.085616,2.222199,11.638692,1.710274e-02
1,BERRIES,1543,4.804947,4.199069,5.498245,6.562774e-113



Top 10 negative associations:


,COMMODITY_DESC,households,odds_ratio,ci_lower,ci_upper,holm_p_value
90,SALAD BAR,1153,0.269285,0.166836,0.434646,1.314491e-05
25,PWDR/CRYSTL DRNK MX,1268,0.328953,0.274200,0.394640,1.191271e-30
50,BAG SNACKS,2348,0.493465,0.415110,0.586610,2.458687e-13
77,ROLLS,1231,0.511420,0.412239,0.634463,1.963725e-07
82,SNACK NUTS,1183,0.524840,0.420382,0.655253,2.191471e-06
54,BREAKFAST SWEETS,1565,0.528744,0.450679,0.620331,1.089411e-12
99,SPICES & EXTRACTS,1904,0.561238,0.448156,0.702854,7.736160e-05
24,FRZN MEAT/MEAT DINNERS,2069,0.585887,0.536947,0.639287,7.089321e-31
64,BATH TISSUES,1852,0.615474,0.534686,0.708468,2.665112e-09
116,TURKEY,972,0.625152,0.489732,0.798018,2.305305e-02


#### Prepare the Sensitivity Analysis

Create a sensitivity sample using all categories with valid numerical estimates and non-empty outcome groups, without applying the 30-household minimum. This checks whether the primary findings depend heavily on the household-count threshold.

In [47]:
# Prepare the sensitivity analysis sample
rq4_sensitivity_results = rq4_results_checked[
    rq4_results_checked["valid_estimate"]
    & rq4_results_checked["COMMODITY_DESC"].isin(
        outcome_support[outcome_support].index
    )
].copy()

# Apply Holm correction to the sensitivity sample
sensitivity_reject, sensitivity_pvalues, _, _ = multipletests(
    rq4_sensitivity_results["p_value"],
    alpha=0.05,
    method="holm"
)

rq4_sensitivity_results["holm_p_value"] = sensitivity_pvalues
rq4_sensitivity_results["significant_after_holm"] = sensitivity_reject

# Summarize sensitivity results
print("Categories in sensitivity analysis:",
      len(rq4_sensitivity_results))

print("Significant after Holm correction:",
      rq4_sensitivity_results["significant_after_holm"].sum())

print("Significant positive associations:",
      (
          rq4_sensitivity_results["significant_after_holm"]
          & (rq4_sensitivity_results["odds_ratio"] > 1)
      ).sum())

print("Significant negative associations:",
      (
          rq4_sensitivity_results["significant_after_holm"]
          & (rq4_sensitivity_results["odds_ratio"] < 1)
      ).sum())

Categories in sensitivity analysis: 261
Significant after Holm correction: 124
Significant positive associations: 110
Significant negative associations: 14


#### RQ4: Interpretation

The analysis examined whether the association between discounts and multi-unit purchasing differed across commodity categories. Separate household-clustered GEE models were fitted for 258 eligible categories, followed by Holm correction to account for multiple testing.

After correction, 124 categories showed statistically significant associations. In 110 categories, discounted purchases were associated with higher odds of multi-unit purchasing, while 14 categories showed lower odds. The remaining 134 categories had no statistically significant association. The median odds ratio across the categories was 1.57.

The strength and direction of the association varied considerably across categories. For example, categories such as Berries and Fluid Milk Products showed higher odds of multi-unit purchasing when discounted, whereas Salad Bar and Bag Snacks showed lower odds.

A sensitivity analysis including 261 categories without the minimum 30-household threshold produced exactly the same number and direction of significant associations. This supports the robustness of the findings to the household-count threshold.

Overall, discounts were positively associated with multi-unit purchasing in most categories, but this relationship was not universal. These results describe associations and do not establish that discounts caused customers to purchase multiple units.

In [48]:
########

#### Prepare Statistical Analysis Results for Export

Collect the final result tables and model estimates from RQ1–RQ4 into separate DataFrames. Include the key test statistics, odds ratios, confidence intervals, adjusted p-values and sensitivity results. This uses the existing analysis objects without fitting any models again.

In [49]:
# RQ1: Bootstrap and CMH test results
rq1_results_export = pd.DataFrame({
    "Measure": [
        "Discounted multi-unit rate",
        "Non-discounted multi-unit rate",
        "Difference (percentage points)",
        "Bootstrap CI lower",
        "Bootstrap CI upper",
        "CMH statistic",
        "CMH p-value",
        "Common odds ratio",
        "Common OR CI lower",
        "Common OR CI upper"
    ],
    "Value": [
        discounted_rate,
        non_discounted_rate,
        observed_difference,
        confidence_interval[0],
        confidence_interval[1],
        test_result.statistic,
        test_result.pvalue,
        cmh_result.oddsratio_pooled,
        cmh_result.oddsratio_pooled_confint()[0],
        cmh_result.oddsratio_pooled_confint()[1]
    ]
})

# RQ2: GEE model estimates
rq2_results_export = pd.DataFrame({
    "Coefficient": gee_result_ind.params.index,
    "Estimate": gee_result_ind.params.values,
    "Standard_Error": gee_result_ind.bse.values,
    "P_Value": gee_result_ind.pvalues.values,
    "Odds_Ratio": np.exp(gee_result_ind.params.values)
})

# RQ3: GEE model estimates
rq3_results_export = pd.DataFrame({
    "Coefficient": gee_rq3_result.params.index,
    "Estimate": gee_rq3_result.params.values,
    "Standard_Error": gee_rq3_result.bse.values,
    "P_Value": gee_rq3_result.pvalues.values,
    "Odds_Ratio": np.exp(gee_rq3_result.params.values)
})

# RQ4: Final primary and sensitivity results
rq4_primary_export = rq4_primary_results.copy()
rq4_sensitivity_export = rq4_sensitivity_results.copy()

# Overall summary of the four research questions
statistical_summary = pd.DataFrame({
    "Research_Question": ["RQ1", "RQ2", "RQ3", "RQ4"],
    "Main_Result": [
        "Discounted purchases had a higher multi-unit purchasing rate",
        "Multi-unit purchasing varied across discount bands",
        "Discount association differed slightly between brand types",
        "Discount association varied across commodity categories"
    ],
    "Key_Result": [
        f"Difference: {observed_difference:.2f} percentage points",
        f"Overall Wald p-value: {overall_test_ind.pvalue:.6g}",
        f"Interaction p-value: {interaction_test.pvalue:.6g}",
        f"{rq4_primary_results['significant_after_holm'].sum()} significant categories out of {len(rq4_primary_results)}"
    ]
})

print("Prepared export tables:")
print("RQ1:", rq1_results_export.shape)
print("RQ2:", rq2_results_export.shape)
print("RQ3:", rq3_results_export.shape)
print("RQ4 primary:", rq4_primary_export.shape)
print("RQ4 sensitivity:", rq4_sensitivity_export.shape)

display(statistical_summary)

Prepared export tables:
RQ1: (10, 2)
RQ2: (5, 5)
RQ3: (4, 5)
RQ4 primary: (258, 12)
RQ4 sensitivity: (261, 12)


,Research_Question,Main_Result,Key_Result
0,RQ1,Discounted purchases had a higher multi-unit p...,Difference: 7.83 percentage points
1,RQ2,Multi-unit purchasing varied across discount b...,Overall Wald p-value: 0
2,RQ3,Discount association differed slightly between...,Interaction p-value: 0.0316435
3,RQ4,Discount association varied across commodity c...,124 significant categories out of 258


#### Export Statistical Analysis Results

Create a dedicated output directory and save the prepared RQ1–RQ4 result tables as CSV files. Also save the overall research-question summary. These files will be reused in the final report, dashboard and subsequent analysis without rerunning the statistical models.

In [50]:
# Create output directory
import os

output_dir = "../data/processed/statistical_analysis"
os.makedirs(output_dir, exist_ok=True)

# Save individual research question results
rq1_results_export.to_csv(
    os.path.join(output_dir, "rq1_overall_discount_analysis.csv"),
    index=False
)

rq2_results_export.to_csv(
    os.path.join(output_dir, "rq2_discount_band_analysis.csv"),
    index=False
)

rq3_results_export.to_csv(
    os.path.join(output_dir, "rq3_brand_comparison.csv"),
    index=False
)

rq4_primary_export.to_csv(
    os.path.join(output_dir, "rq4_category_primary_results.csv"),
    index=False
)

rq4_sensitivity_export.to_csv(
    os.path.join(output_dir, "rq4_category_sensitivity_results.csv"),
    index=False
)

# Save overall summary
statistical_summary.to_csv(
    os.path.join(output_dir, "statistical_analysis_summary.csv"),
    index=False
)

# Confirm saved files
print("Statistical analysis results saved successfully!\n")
print("Output directory:", output_dir)

print("\nSaved files:")
for filename in sorted(os.listdir(output_dir)):
    print("-", filename)

Statistical analysis results saved successfully!

Output directory: ../data/processed/statistical_analysis

Saved files:
- rq1_overall_discount_analysis.csv
- rq2_discount_band_analysis.csv
- rq3_brand_comparison.csv
- rq4_category_primary_results.csv
- rq4_category_sensitivity_results.csv
- statistical_analysis_summary.csv
